In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split

In [2]:
data = pd.read_csv("online_retail_II.csv")

In [3]:
data.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
data = data[data["Quantity"] > 0]

In [5]:
data.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [6]:
nlargestdata = data.groupby('StockCode')['Quantity'].sum().nlargest(250)
nlargestdata = pd.DataFrame(nlargestdata)
nlargestdata = nlargestdata.reset_index()

In [7]:
nlargestdata.info()

<class 'pandas.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   StockCode  250 non-null    str  
 1   Quantity   250 non-null    int64
dtypes: int64(1), str(1)
memory usage: 4.0 KB


In [8]:
nlargestdata.shape

(250, 2)

In [9]:
nlargestdata.isnull().sum()

StockCode    0
Quantity     0
dtype: int64

In [10]:
nlargestdata.describe()

,Quantity
count,250.000000
mean,21156.576000
std,16004.118806
min,10128.000000
25%,12687.000000
50%,15804.000000
75%,22655.250000
max,110249.000000


In [11]:
data["Country"].unique()

<StringArray>
[      'United Kingdom',               'France',                  'USA',
              'Belgium',            'Australia',                 'EIRE',
              'Germany',             'Portugal',              'Denmark',
          'Netherlands',               'Poland',      'Channel Islands',
                'Spain',               'Cyprus',               'Greece',
               'Norway',              'Austria',               'Sweden',
 'United Arab Emirates',              'Finland',                'Italy',
          'Switzerland',                'Japan',          'Unspecified',
              'Nigeria',                'Malta',              'Bahrain',
                  'RSA',              'Bermuda',            'Hong Kong',
            'Singapore',             'Thailand',               'Israel',
            'Lithuania',          'West Indies',              'Lebanon',
                'Korea',               'Brazil',               'Canada',
              'Iceland',         'Sau

In [12]:
newdata = data[data["StockCode"].isin(nlargestdata['StockCode'])]

In [13]:
newdata.shape

(282214, 8)

In [14]:
newdata.isnull().sum()

Invoice            0
StockCode          0
Description      149
Quantity           0
InvoiceDate        0
Price              0
Customer ID    52778
Country            0
dtype: int64

In [15]:
newdata = newdata.dropna(subset=["Description"])

In [16]:
newdata.isnull().sum()

Invoice            0
StockCode          0
Description        0
Quantity           0
InvoiceDate        0
Price              0
Customer ID    52629
Country            0
dtype: int64

In [17]:
newdata.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
15,489436,84879,ASSORTED COLOUR BIRD ORNAMENT,16,2009-12-01 09:06:00,1.69,13078.0,United Kingdom
25,489436,21181,PLEASE ONE PERSON METAL SIGN,12,2009-12-01 09:06:00,2.10,13078.0,United Kingdom
30,489436,22111,SCOTTIE DOG HOT WATER BOTTLE,24,2009-12-01 09:06:00,4.25,13078.0,United Kingdom


In [18]:
newdata['InvoiceDate'] = pd.to_datetime(newdata['InvoiceDate']).dt.date

In [19]:
newdata.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01,1.25,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01,1.25,13085.0,United Kingdom
15,489436,84879,ASSORTED COLOUR BIRD ORNAMENT,16,2009-12-01,1.69,13078.0,United Kingdom
25,489436,21181,PLEASE ONE PERSON METAL SIGN,12,2009-12-01,2.10,13078.0,United Kingdom
30,489436,22111,SCOTTIE DOG HOT WATER BOTTLE,24,2009-12-01,4.25,13078.0,United Kingdom


In [20]:
filtereddata = newdata.groupby(["InvoiceDate", "StockCode"])["Quantity"].sum().reset_index()

In [21]:
filtereddata.head()

,InvoiceDate,StockCode,Quantity
0,2009-12-01,15034,3
1,2009-12-01,15036,55
2,2009-12-01,15056N,30
3,2009-12-01,16047,1
4,2009-12-01,16156S,25


In [22]:
filtereddata.shape

(92673, 3)

In [23]:
def fill_dates(group):
    date_range = pd.date_range(group['InvoiceDate'].min(), group['InvoiceDate'].max(), freq='D')

    group = group.set_index('InvoiceDate').reindex(date_range, fill_value=0)
    group.index.name = 'InvoiceDate'
    return group

full_data = filtereddata.groupby('StockCode').apply(fill_dates).reset_index()


In [24]:
full_data

,StockCode,InvoiceDate,Quantity
0,15034,2009-12-01,3
1,15034,2009-12-02,0
2,15034,2009-12-03,0
3,15034,2009-12-04,0
4,15034,2009-12-05,0
...,...,...,...
159904,POST,2011-12-05,15
159905,POST,2011-12-06,25
159906,POST,2011-12-07,21
159907,POST,2011-12-08,12


In [25]:
full_data = full_data.sort_values(['StockCode', 'InvoiceDate'])

In [26]:
full_data

,StockCode,InvoiceDate,Quantity
0,15034,2009-12-01,3
1,15034,2009-12-02,0
2,15034,2009-12-03,0
3,15034,2009-12-04,0
4,15034,2009-12-05,0
...,...,...,...
159904,POST,2011-12-05,15
159905,POST,2011-12-06,25
159906,POST,2011-12-07,21
159907,POST,2011-12-08,12


In [31]:
full_data['lag_1'] = full_data.groupby('InvoiceDate')['StockCode'].shift(1)

In [42]:
full_data.head()

,StockCode,InvoiceDate,Quantity,lag_1
0,15034,2009-12-01,3,NaN
1,15034,2009-12-02,0,NaN
2,15034,2009-12-03,0,NaN
3,15034,2009-12-04,0,NaN
4,15034,2009-12-05,0,NaN
